# HyperRAG Tutorial: Step-by-Step Pipeline Walkthrough

**HyperRAG** is a Link-Graph Augmented Retrieval-Augmented Generation (RAG) system for multi-hop web question answering. Unlike standard RAG, which retrieves a fixed set of documents and stops there, HyperRAG exploits the native HTML hyperlinks between Wikipedia pages to *expand* retrieval into adjacent pages — following the same kind of reasoning path a human researcher would take when clicking links to find related information.

This tutorial walks through the complete HyperRAG pipeline in four stages:

1. **Stage 1 — Corpus Loading**: Load multi-hop questions from HotpotQA and build a corpus of Wikipedia pages.
2. **Stage 2 — Graph Construction**: Build a directed hyperlink graph (NetworkX DiGraph) from the `<a href>` links in the Wikipedia HTML pages.
3. **Stage 3 — Embeddings & FAISS Index**: Encode every corpus page as a dense vector using `sentence-transformers/all-MiniLM-L6-v2` and store them in a FAISS flat index for fast similarity search.
4. **Stage 4 — Retrieval Systems + Evaluation**: Compare three systems — B1 (Naive RAG), B2 (HtmlRAG-style), and HyperRAG — on real HotpotQA questions using Exact Match (EM) and F1 metrics.

**Who is this for?** Graders wanting to understand how each component works, and developers wanting to replicate or extend the system.

**What makes HyperRAG different?** Standard RAG and HtmlRAG retrieve pages independently. HyperRAG additionally follows the hyperlinks FROM those retrieved pages one hop outward — discovering pages the query didn't directly match but that a human reader following links would naturally reach. This 1-hop graph expansion is the core innovation, and it costs almost nothing: the hyperlink graph already *exists* inside the HTML `<a href>` tags. No LLM-extracted knowledge graph construction is needed.

## Setup

**Google Colab users:** Run the pip install cell below first, then run the imports cell.

**Local users:** Skip the pip install cell — all dependencies are already installed via `requirements.txt`.

In [ ]:
# Uncomment in Google Colab:
# !pip install -q transformers sentence-transformers faiss-cpu networkx datasets beautifulsoup4 requests numpy torch tqdm matplotlib

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path so we can import from src/
# Works whether running from notebooks/ dir (local) or the project root (Colab)
PROJECT_ROOT = Path('..') if Path('../src').exists() else Path('.')
sys.path.insert(0, str(PROJECT_ROOT))

# Stage 1 imports: HotpotQA loading and Wikipedia corpus management
from src.corpus import load_hotpotqa, load_corpus, build_corpus

# Stage 2 imports: NetworkX DiGraph construction and persistence
from src.graph import build_graph, build_and_save_graph, load_graph, print_graph_stats

# Stage 3 imports: sentence-transformers encoding and FAISS index
from src.embeddings import build_faiss_index, load_index, search

# Stage 4 imports: retrieval systems (B1, B2, HyperRAG) and evaluation metrics
from src.retrieval import naive_rag, htmlrag_style, hyperrag, compute_em, compute_f1

# Visualization and numeric libraries
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import numpy as np

print("All imports successful!")

## Stage 1: Corpus Loading

**What is HotpotQA?** HotpotQA (Yang et al., 2018) is a multi-hop question answering benchmark. Unlike simple QA datasets where a single paragraph contains the answer, HotpotQA questions require reasoning across *two or more* Wikipedia articles. For example, answering "What film starred the actor who played Moriarty in BBC Sherlock and was featured at Cannes 2003?" requires finding who played Moriarty, then finding what films that actor appeared in, then checking the Cannes film list — three hops of reasoning.

**Why do we need a corpus?** A RAG system doesn't answer questions from scratch; it retrieves relevant documents from a pre-built *document store*, then constructs an answer from the retrieved text. Our corpus is the document store: a collection of ~450 Wikipedia pages referenced by the first 500 HotpotQA questions. Without this corpus, there is nothing to retrieve from.

**What does each corpus page contain?** Each page is a Python dict with four fields:
- `title`: The Wikipedia article title (e.g. `"Albert Einstein"`).
- `text`: The plain text of the page (stripped of HTML tags). Used by B1 Naive RAG.
- `html`: The raw HTML of the page. Used by B2 HtmlRAG-style to preserve structure (tables, headings).
- `links`: A list of page titles this page links TO via `<a href>` tags. This is the raw material for the hyperlink graph in Stage 2.

**The caching pattern:** Fetching 450 Wikipedia pages from the Wikipedia API takes 20–40 minutes. To avoid re-fetching every session, `src/corpus.py` saves the result to `data/corpus.json` and checks for it on startup. If it exists, the corpus loads in under a second.

### Step 1.1: Load HotpotQA Questions

We load a small subset of HotpotQA training questions. Each item has a `question`, a gold `answer`, and `supporting_facts` — a list of (Wikipedia title, sentence index) pairs pointing to the exact Wikipedia pages and sentences the question was constructed from. The supporting facts tell us which pages are *relevant* to each question, which is how we compute EM/F1 at evaluation time.

In [ ]:
# Load 3 HotpotQA questions for our tutorial walkthrough
# We use n_samples=3 to keep execution fast — the full eval uses 50 questions
qa_items = load_hotpotqa(split="train", n_samples=3)

# Show the structure of each QA item
print(f"Loaded {len(qa_items)} questions\n")
for i, qa in enumerate(qa_items):
    print(f"Q{i+1}: {qa['question']}")
    print(f"    Answer: {qa['answer']}")
    # supporting_facts['title'] is a list of Wikipedia titles whose sentences support the answer
    # These are the pages HyperRAG must retrieve to score EM=1 on this question
    sf_titles = list(set(sf for sf in qa['supporting_facts']['title']))
    print(f"    Supporting pages: {sf_titles}\n")

### Step 1.2: Load the Corpus

The corpus was pre-built by `src/corpus.py` during Phase 2 of development. The build process:
1. Loaded the first 500 HotpotQA questions from the `train` split.
2. Extracted all unique Wikipedia page titles mentioned in `supporting_facts` across those questions.
3. For each title, fetched the page from the Wikipedia API using the `parse` action with `prop=text|links`.
4. Parsed the returned HTML with BeautifulSoup to extract plain text and outgoing hyperlinks.
5. Saved the result incrementally to `data/corpus.json` (every 50 pages) to protect against network interruptions.

The result is approximately 450 pages covering the Wikipedia topics needed to answer 500 HotpotQA multi-hop questions. Below we load this pre-built corpus and inspect its structure.

In [ ]:
# Use pathlib for all file paths — no hardcoded strings (CLAUDE.md requirement)
CORPUS_PATH = PROJECT_ROOT / "data" / "corpus.json"

# Load pre-built corpus (built by src/corpus.py in Phase 2)
# If data/corpus.json doesn't exist, run: python -c "from src.corpus import build_corpus; build_corpus()"
corpus = load_corpus(CORPUS_PATH)
print(f"Corpus: {len(corpus)} Wikipedia pages\n")

# Inspect the structure of one page to understand what each field contains
page = corpus[0]
print(f"Page title:      {page['title']}")
print(f"Text length:     {len(page['text'])} characters (plain text, used by B1)")
print(f"HTML length:     {len(page['html'])} characters (raw HTML, used by B2)")
print(f"Outgoing links:  {len(page['links'])} links (used to build Stage 2 graph)")
print(f"First 5 links:   {page['links'][:5]}")
print(f"\nText preview (first 300 chars):\n{page['text'][:300]}...")

### Stage 1 Summary

We now have two things: a set of HotpotQA questions with gold answers and supporting page references, and a Wikipedia corpus of ~450 pages each containing plain text, raw HTML, and outgoing hyperlinks. The plain text and HTML will be used by the retrieval systems in Stage 4. The outgoing hyperlinks are the raw material for Stage 2: building the graph that powers HyperRAG's 1-hop expansion.

## Stage 2: Graph Construction

**What is a directed graph (DiGraph)?** A directed graph is a set of *nodes* (vertices) connected by *directed edges* (arrows). An edge from node A to node B means "A points to B", but does NOT imply an edge from B back to A. In our hyperlink graph:
- **Nodes** are Wikipedia page titles (e.g. `"Albert Einstein"`).
- **Edges** are hyperlinks: if the Wikipedia page for A contains a `<a href>` link to B, we add a directed edge A → B.

**Why directed?** Hyperlinks are asymmetric. The Wikipedia page for "Theory of Relativity" might link to "Albert Einstein", but the page for "Albert Einstein" might also link back — or might not. The link direction matters because HyperRAG uses *outgoing* links (successors) to expand from a retrieved page to pages it references, not the other way around. Using an undirected graph would conflate these two different relationships.

**Why does this graph matter for HyperRAG?** Multi-hop questions require finding connections between concepts that no single page explicitly states. When we retrieve a page via semantic search, that page's outgoing hyperlinks often point to the *next* page the question requires. HyperRAG follows those links one hop outward — exactly as a human researcher would click to the linked page to continue researching. We call this the "1-hop graph expansion": starting from k semantically similar pages, expand to all their successors in the DiGraph, then re-rank the combined set.

**How we build it:** `src/graph.py` uses a two-pass algorithm. Pass 1: add all corpus page titles as nodes. Pass 2: for each page, iterate through its `links` list; if a link target is also a node in the graph (i.e., also in our corpus), add a directed edge. We only add edges between pages we have fetched — we cannot follow links to pages outside our corpus.

In [ ]:
# Build directed graph from corpus hyperlinks
# Only edges between pages WITHIN our corpus are created
# (We can't follow links to pages we haven't fetched)
graph = build_graph(corpus)

# print_graph_stats shows node count, edge count, density, and top-degree nodes
print_graph_stats(graph)

print(f"\nGraph type:   {type(graph).__name__}  (DiGraph = directed)")
print(f"Is directed:  {graph.is_directed()}")

### Step 2.1: Exploring Graph Structure

We can inspect individual nodes to understand their connectivity. The most important operation for HyperRAG is `G.successors(node)`, which returns all pages that a given page links TO. These are the "1-hop expansion targets": when HyperRAG retrieves a page, it calls `G.successors(page_title)` to discover which pages should be added to the candidate set. The complementary operation is `G.predecessors(node)`, which returns pages that link TO this page — useful for understanding which pages are heavily cited across the corpus.

In [ ]:
# Find a page that has at least 2 outgoing links within our corpus
# (some pages link only to pages not in our subset, so we search)
example_node = None
for node in graph.nodes():
    successors = list(graph.successors(node))
    if len(successors) >= 2:  # need at least 2 for an interesting visualization
        example_node = node
        break

if example_node:
    successors = list(graph.successors(example_node))
    predecessors = list(graph.predecessors(example_node))
    print(f"Page: '{example_node}'")
    print(f"  Links TO (successors):   {successors[:10]}")
    print(f"  Links FROM (predecessors): {predecessors[:10]}")
    print(f"\n  Out-degree: {graph.out_degree(example_node)}  (outgoing hyperlinks to other corpus pages)")
    print(f"  In-degree:  {graph.in_degree(example_node)}  (other corpus pages that link here)")
    # This asymmetry is WHY we use a DiGraph — hyperlinks are not symmetric
    print(f"\n  Note: out-degree != in-degree in general — hyperlinks are NOT symmetric")
else:
    print("No node with 2+ successors found — corpus may be very small")

### Step 2.2: Visualizing the Graph Structure

The visualization below shows a *subgraph* centered on an example Wikipedia page and its immediate neighbors in the hyperlink graph. Each node is a Wikipedia page title; each arrow is a directed hyperlink. We use three colors:

- **Red** — the center page (the one our semantic search retrieved for a query).
- **Orange** — 1-hop successors: pages the center page links TO. These are the pages HyperRAG adds to the candidate set during graph expansion. This is the core HyperRAG operation.
- **Blue** — predecessors: pages that link TO the center page. These are NOT part of the HyperRAG expansion (we follow outgoing links, not incoming links), but they show how the page fits into the broader Wikipedia citation network.

The key insight to take from this visualization: by following just one hop of outgoing links from the red node, we immediately discover the orange nodes — pages that may contain the "second half" of a multi-hop answer that a direct semantic search would never retrieve.

In [ ]:
# --- Graph Structure Visualization ---
# Show a subgraph centered on example_node and its immediate neighbors

if example_node:
    # Collect the subgraph: center + up to 6 successors + up to 6 predecessors
    succ_set = set(list(graph.successors(example_node))[:6])
    pred_set = set(list(graph.predecessors(example_node))[:6])
    neighbors = succ_set | pred_set | {example_node}
    subgraph = graph.subgraph(neighbors)

    fig, ax = plt.subplots(figsize=(12, 8))

    # Color code: center=red, successors=orange (HyperRAG expansion targets), predecessors=blue
    node_colors = []
    for n in subgraph.nodes():
        if n == example_node:
            node_colors.append("#e74c3c")   # red — retrieved/center page
        elif n in succ_set:
            node_colors.append("#f39c12")   # orange — 1-hop expansion targets (HyperRAG adds these)
        else:
            node_colors.append("#3498db")   # blue — predecessors (link to center, not expanded)

    # Deterministic spring layout so the diagram is reproducible
    pos = nx.spring_layout(subgraph, seed=42, k=2)

    # Draw with directed arrows to make the hyperlink direction visible
    nx.draw(
        subgraph, pos, ax=ax,
        node_color=node_colors,
        node_size=1500,
        font_size=7,
        font_weight="bold",
        with_labels=True,
        arrows=True,
        edge_color="#cccccc",
        arrowsize=15,
        connectionstyle="arc3,rad=0.1",  # slight curve so bidirectional edges are visible
    )

    # Legend explaining the color scheme
    legend_elements = [
        mpatches.Patch(color="#e74c3c", label="Center page (retrieved by semantic search)"),
        mpatches.Patch(color="#f39c12", label="1-hop successors (HyperRAG expansion targets)"),
        mpatches.Patch(color="#3498db", label="Predecessors (link TO center — not expanded)"),
    ]
    ax.legend(handles=legend_elements, loc="upper left", fontsize=9)
    ax.set_title(f"Hyperlink Graph Subgraph Around '{example_node}'", fontsize=13)

    plt.tight_layout()
    plt.show()  # inline display only — no file saving (D-06)
else:
    print("No suitable node found for visualization — corpus may be too small")

### Step 2.3: Full Pipeline Overview

Before we move to Stage 3 (Embeddings & FAISS), here is the complete HyperRAG pipeline at a glance. The four stages are sequential: each stage produces output that the next stage consumes. The pipeline is designed so that expensive stages (corpus fetching in Stage 1, index building in Stage 3) are cached to disk and only run once.

In [ ]:
# --- Pipeline Flow Diagram ---
# Matplotlib diagram showing the 4 stages as labeled boxes with directional arrows
# No external diagram libraries needed — pure matplotlib patches

fig, ax = plt.subplots(figsize=(14, 3))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3)
ax.axis("off")  # hide axes — we're drawing manually

# Each tuple: (x_center, box_width, main_label, subtitle_label)
stages = [
    (1.5,  2.5, "Stage 1\nCorpus Loading",      "HotpotQA + Wikipedia"),
    (5.0,  2.5, "Stage 2\nGraph Construction",   "NetworkX DiGraph"),
    (8.5,  2.5, "Stage 3\nEmbeddings + FAISS",   "Vector similarity search"),
    (12.0, 2.5, "Stage 4\nRetrieval + Eval",      "B1 / B2 / HyperRAG"),
]

# One distinct color per stage so graders can map prose to diagram at a glance
colors = ["#3498db", "#2ecc71", "#e67e22", "#e74c3c"]

for (x, w, label, subtitle), color in zip(stages, colors):
    # Draw rounded rectangle for each stage
    rect = mpatches.FancyBboxPatch(
        (x - w / 2, 0.5), w, 1.8,
        boxstyle="round,pad=0.15",
        facecolor=color, edgecolor="white",
        alpha=0.85, linewidth=2,
    )
    ax.add_patch(rect)
    # Main stage label (bold)
    ax.text(x, 1.6, label, ha="center", va="center",
            fontsize=10, fontweight="bold", color="white")
    # Subtitle / technology note (italic)
    ax.text(x, 0.85, subtitle, ha="center", va="center",
            fontsize=8, color="white", style="italic")

# Arrows connecting stages left-to-right
for i in range(3):
    x_start = stages[i][0] + stages[i][1] / 2 + 0.1    # right edge of box i
    x_end   = stages[i+1][0] - stages[i+1][1] / 2 - 0.1  # left edge of box i+1
    ax.annotate(
        "", xy=(x_end, 1.4), xytext=(x_start, 1.4),
        arrowprops=dict(arrowstyle="->", color="#333333", lw=2),
    )

ax.set_title("HyperRAG Pipeline: From Raw Data to Evaluation", fontsize=13, pad=15)
plt.tight_layout()
plt.show()  # inline display only — no file saving (D-06)

### Stage 2 Summary

We now have a directed hyperlink graph connecting Wikipedia pages in our corpus. The key structural insight is that this graph's edges encode the same navigational paths a human researcher would follow: starting from a retrieved page, follow its outgoing links (successors in the DiGraph) to discover adjacent pages that contain complementary information. This is exactly what HyperRAG does in Stage 4 — retrieve k pages via dense vector search, then expand to their successors, then re-rank the combined set.

In Stage 3, we will encode every corpus page as a dense vector so that we can find the k pages most semantically similar to a query in milliseconds — regardless of whether the query contains the same words as the page text.

## Stage 3: Embeddings & FAISS Vector Index

**What are embeddings?** An embedding is a fixed-length numerical vector that represents a piece of text. We use the `sentence-transformers/all-MiniLM-L6-v2` model, which converts any text into a 384-dimensional vector. Pages about similar topics produce vectors that point in similar directions in this high-dimensional space — "Albert Einstein" and "Theory of Relativity" end up close together, while "Albert Einstein" and "2003 Cannes Film Festival" end up far apart.

**What is FAISS?** FAISS (Facebook AI Similarity Search) is a library optimised for fast nearest-neighbour lookups in vector spaces. Instead of comparing a query vector against every document individually (O(n) per query), FAISS uses optimised data structures to find the most similar vectors in sub-linear time. For our ~450-page corpus, brute-force would be fast enough, but FAISS scales to millions of vectors without code changes.

**How does cosine similarity work here?** Cosine similarity measures the angle between two vectors — 1.0 means identical direction (very similar content), 0.0 means perpendicular (unrelated). We use **L2 normalisation + inner product** in FAISS, which is mathematically equivalent to cosine similarity but avoids a separate cosine index type. Every vector is normalised to unit length before indexing; then inner product equals cosine similarity automatically.

**Why does this matter for HyperRAG?** FAISS retrieval is Stage 4's starting point for all three systems. B1 and B2 stop here; HyperRAG uses FAISS to find an initial candidate set and then expands it via graph expansion before re-ranking.

In [ ]:
# Build FAISS index: encode all corpus pages as 384-dim vectors
# model: all-MiniLM-L6-v2 — fast on CPU, adequate quality for prototype
# Internally: L2-normalises each vector so inner product = cosine similarity
index, page_ids = build_faiss_index(corpus)

print("FAISS index built:")
print(f"  Vectors indexed:  {index.ntotal}")         # one vector per corpus page
print(f"  Dimensions:       {index.d}")               # 384 for MiniLM-L6-v2
print(f"  Page IDs (titles): {len(page_ids)} entries")  # maps row position → page title
print(f"  Index type:        {type(index).__name__}")  # IndexFlatIP — flat inner-product

### Step 3.1: Searching the Index

Given a natural-language question, we encode it into the same 384-dimensional space as our corpus pages and find the nearest neighbours. The `search()` function returns a list of corpus page dicts sorted by cosine similarity — the most semantically similar pages appear first. Notice that the search uses the question text directly (no need to identify keywords): the embedding model maps both question and page text into the same semantic space, so even paraphrases and indirect references can match.

In [ ]:
# Use the first question from our live HotpotQA subset
example_question = qa_items[0]["question"]
example_answer   = qa_items[0]["answer"]
print(f"Query:       {example_question}")
print(f"Gold answer: {example_answer}\n")

# Search for the top-5 most semantically similar pages
top_pages = search(example_question, index, page_ids, corpus, k=5)

# Display results — flag any page that already contains the gold answer
print("Top-5 retrieved pages:")
for i, page in enumerate(top_pages):
    # Retrieval EM: does the gold answer appear in this page's plain text?
    contains_answer = example_answer.lower() in page["text"].lower()
    marker = "  ← CONTAINS ANSWER" if contains_answer else ""
    print(f"  {i+1}. {page['title']}{marker}")
    print(f"     Preview: {page['text'][:120].strip()}...")

### Step 3.2: Visualizing Similarity Scores

The bar chart below shows the cosine similarity score between our query and each of the top-10 retrieved pages. Scores range from 0.0 (no similarity) to 1.0 (identical meaning). Notice how scores typically drop off quickly after the top few results — this drop-off is why choosing the right value of `k` matters: too small and you miss relevant pages, too large and you dilute the context with irrelevant noise. The yellow dashed line marks the lowest score among the top-5 (the threshold for B1/B2 retrieval).

In [ ]:
# --- Similarity Score Bar Chart ---
# Encode the query and run a raw FAISS search to get cosine similarity scores
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Encode query to 384-dim vector
query_vec = model.encode([example_question])
# L2-normalise so inner product = cosine similarity (same as build_faiss_index does internally)
query_norm = query_vec / np.linalg.norm(query_vec, axis=1, keepdims=True)

# Search top-10 for a richer visualization
distances, indices_found = index.search(query_norm.astype(np.float32), k=10)
scores = distances[0]           # cosine similarity scores (already normalised)
titles = [page_ids[idx] for idx in indices_found[0]]

# Truncate long titles so bars fit in the chart
short_titles = [t[:38] + "…" if len(t) > 38 else t for t in titles]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(range(len(scores)), scores, color="#3498db", edgecolor="white")

ax.set_yticks(range(len(scores)))
ax.set_yticklabels(short_titles, fontsize=9)
ax.invert_yaxis()  # highest score at the top

ax.set_xlabel("Cosine Similarity Score", fontsize=11)
ax.set_title(
    f"Top-10 Pages by Similarity to:\n\"{example_question[:85]}{'…' if len(example_question)>85 else ''}\"",
    fontsize=10,
)

# Add score value label at the end of each bar
for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
            f"{score:.3f}", va="center", fontsize=9)

# Mark the k=5 threshold (the cutoff used by B1/B2)
if len(scores) >= 5:
    threshold = scores[4]  # 5th-highest score
    ax.axvline(threshold, color="#f1c40f", linestyle="--", linewidth=1.5,
               label=f"k=5 cutoff ({threshold:.3f})")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()  # inline display only — no file saving

### Stage 3 Summary

We now have a FAISS vector index that finds the most semantically similar corpus pages for any query in milliseconds — regardless of exact wording. However, FAISS similarity alone answers only the "what is directly related to the question?" part. Multi-hop questions often require a second page that is *not* directly similar to the question but is linked *from* a directly similar page. That reasoning step is what Stage 4's HyperRAG system adds on top of FAISS.

## Stage 4: Retrieval Systems & Evaluation

The core question of this project is: *given a multi-hop question, which retrieval approach finds the most relevant context?* We compare three systems on the same set of HotpotQA questions:

- **B1 — Naive RAG:** Encode the question, search FAISS for the top-k pages, concatenate their **plain text**. This is the simplest possible RAG baseline. Its weakness: it can only find pages whose text is directly similar to the question. If the answer is on a page that is topically distant from the question but reachable by following hyperlinks, B1 misses it entirely.

- **B2 — HtmlRAG-style:** Same FAISS retrieval as B1, but the output preserves **HTML structural tags** (headings, tables, lists) instead of stripping them. The hypothesis: structural markup helps a downstream reader (human or LLM) parse the context more accurately. B2 tests whether the *format* of the retrieved text matters even when the *set* of pages is identical.

- **HyperRAG (our approach):** Uses FAISS to retrieve top-k pages, then **expands** the candidate set by following outgoing hyperlinks (1-hop graph successors) from each retrieved page. The expanded set is re-ranked by cosine similarity to the query, and the top `expand_k` pages are returned. The core insight: a Wikipedia page about "2003 Cannes Film Festival" might be semantically distant from a question about an actor's birthplace, but if FAISS retrieves it because the actor is mentioned there, HyperRAG will follow its links to the actor's own page — which does contain the birthplace. One hop of graph expansion can close reasoning chains that pure vector search cannot.

**Evaluation metric — Retrieval EM/F1:** We are evaluating *retrieval quality*, not generation quality. No LLM is involved. Exact Match (EM) = 1.0 if the gold answer string appears anywhere in the retrieved context; 0.0 otherwise. F1 = token-level overlap between the gold answer tokens and the context tokens. Both metrics ask: "did we retrieve context that contains the answer?"

### Step 4.1: B1 — Naive RAG

The simplest baseline: retrieve the top-k pages by FAISS similarity, concatenate their plain text. No graph, no HTML.

In [ ]:
# B1: Naive RAG — top-k pages by FAISS similarity, plain text concatenated
context_b1 = naive_rag(example_question, index, corpus, k=5)

# Retrieval EM: 1.0 if the gold answer appears anywhere in the context
em_b1 = compute_em(context_b1, example_answer)
# Retrieval F1: token-level overlap between answer tokens and context tokens
f1_b1 = compute_f1(context_b1, example_answer)

print("B1 — Naive RAG:")
print(f"  Context length:  {len(context_b1):,} characters")
print(f"  EM:   {em_b1:.1f}   |   F1: {f1_b1:.4f}")
print(f"  Answer '{example_answer}' found in context: {'YES ✓' if em_b1 else 'NO ✗'}")
print(f"\n  Context preview (first 300 chars):\n  {context_b1[:300].strip()}…")

### Step 4.2: B2 — HtmlRAG-style

Same retrieval as B1, but the context preserves HTML structural tags (headings, tables, lists). Same pages, different representation.

In [ ]:
# B2: HtmlRAG-style — same FAISS retrieval, but preserves structural HTML tags
# This tests whether the output FORMAT matters even when the retrieved PAGES are the same as B1
context_b2 = htmlrag_style(example_question, index, corpus, k=5)

em_b2 = compute_em(context_b2, example_answer)
f1_b2 = compute_f1(context_b2, example_answer)

print("B2 — HtmlRAG-style:")
print(f"  Context length:  {len(context_b2):,} characters")
print(f"  EM:   {em_b2:.1f}   |   F1: {f1_b2:.4f}")
print(f"  Answer '{example_answer}' found in context: {'YES ✓' if em_b2 else 'NO ✗'}")
print(f"\n  Context preview (first 300 chars):\n  {context_b2[:300].strip()}…")
print(f"\n  (HTML tags are preserved — compare length with B1: {len(context_b2) - len(context_b1):+,} chars)")

### Step 4.3: HyperRAG — Graph-Augmented Retrieval

HyperRAG is our core contribution. Here is the algorithm step by step:

1. **Dense retrieval:** Search FAISS for the top-k pages most similar to the query. This produces the same initial candidate set as B1.
2. **Graph expansion:** For each of the k retrieved pages, call `graph.successors(page_title)` to get all pages it links TO. This discovers pages that the *retrieved page's author thought were relevant* — the kind of pages a human researcher would click to next.
3. **Deduplicate:** Combine the k initial pages and all their successors into a single candidate pool (removing duplicates).
4. **Re-rank:** Encode the query, compute cosine similarity against every page in the candidate pool, and select the top `expand_k` pages by similarity score.
5. **Return:** Concatenate the top `expand_k` pages' plain text as the final context.

The key insight is that graph expansion gives HyperRAG access to pages that are *semantically distant* from the query but *structurally adjacent* to relevant pages via hyperlinks. This is exactly the kind of reasoning that multi-hop questions require — and it costs almost nothing extra: the hyperlinks were already in the HTML.

In [ ]:
# HyperRAG: FAISS retrieval + 1-hop graph expansion + cosine re-ranking
# k=5: initial FAISS candidates; expand_k=5: final pages after graph expansion + re-rank
context_hr = hyperrag(example_question, index, corpus, graph, k=5, expand_k=5)

em_hr = compute_em(context_hr, example_answer)
f1_hr = compute_f1(context_hr, example_answer)

print("HyperRAG:")
print(f"  Context length:  {len(context_hr):,} characters")
print(f"  EM:   {em_hr:.1f}   |   F1: {f1_hr:.4f}")
print(f"  Answer '{example_answer}' found in context: {'YES ✓' if em_hr else 'NO ✗'}")
print(f"\n  Context preview (first 300 chars):\n  {context_hr[:300].strip()}…")

# Quick comparison for this single question
print("\n--- Single-question comparison ---")
print(f"  B1 EM={em_b1:.1f} F1={f1_b1:.4f}  |  B2 EM={em_b2:.1f} F1={f1_b2:.4f}  |  HyperRAG EM={em_hr:.1f} F1={f1_hr:.4f}")

### Step 4.4: Evaluating Across Multiple Questions

A single question is not enough to draw conclusions — results can vary widely depending on whether the answer happens to appear in a directly similar page. We now run all three systems on our full set of 3 tutorial questions and compute average EM and F1. This gives a more reliable picture of each system's retrieval quality.

**EM (Exact Match)** answers "did we retrieve any context that contains the gold answer string?" — a binary 0/1 per question, averaged across questions.

**F1 (Token F1)** measures the overlap between the set of tokens in the gold answer and the set of tokens in the full retrieved context. It is a softer metric: even if the exact answer string is not present, a context that contains most of the answer words will score a partial F1.

In [ ]:
# Evaluate all 3 systems on every question in qa_items (live execution — not pre-computed)
systems_config = [
    ("B1: Naive RAG",   naive_rag,    {}),
    ("B2: HtmlRAG",     htmlrag_style, {}),
    ("HyperRAG",        hyperrag,      {"graph": graph, "expand_k": 5}),
]

results = {}
for name, system_fn, extra_kwargs in systems_config:
    em_scores, f1_scores = [], []
    for qa in qa_items:
        # Run retrieval system on each question
        ctx = system_fn(qa["question"], index, corpus, k=5, **extra_kwargs)
        # Compute retrieval-quality metrics (no LLM inference needed)
        em_scores.append(compute_em(ctx, qa["answer"]))
        f1_scores.append(compute_f1(ctx, qa["answer"]))
    results[name] = {
        "avg_em": sum(em_scores) / len(em_scores),
        "avg_f1": sum(f1_scores) / len(f1_scores),
    }

# Print results table
print(f"{'System':<22}  {'Avg EM':>8}  {'Avg F1':>8}")
print("-" * 44)
for name, scores in results.items():
    print(f"{name:<22}  {scores['avg_em']:>8.3f}  {scores['avg_f1']:>8.3f}")
print(f"\nEvaluated on {len(qa_items)} multi-hop HotpotQA questions")
print("EM  = does gold answer appear in retrieved context? (0 or 1 per question, averaged)")
print("F1  = token-level overlap between gold answer and context")

### Step 4.5: Comparison Visualization

The grouped bar chart below shows both metrics side by side for all three systems. Taller bars are better. If HyperRAG's graph expansion is working as intended, its bars should be equal to or higher than B1/B2 — especially on questions where the answer is on a page reachable only via hyperlinks from a directly similar page.

In [ ]:
# --- B1 vs B2 vs HyperRAG Comparison Chart ---

system_labels = list(results.keys())
em_vals = [results[s]["avg_em"] for s in system_labels]
f1_vals = [results[s]["avg_f1"] for s in system_labels]

x = np.arange(len(system_labels))  # x positions for each system
width = 0.35                        # width of each bar group

fig, ax = plt.subplots(figsize=(9, 5))

# Two bar groups: EM (blue) and F1 (red)
bars_em = ax.bar(x - width / 2, em_vals, width,
                 label="Exact Match (EM)", color="#3498db", edgecolor="white")
bars_f1 = ax.bar(x + width / 2, f1_vals, width,
                 label="F1 Score",         color="#e74c3c", edgecolor="white")

# Label each bar with its numeric value
for bar in bars_em:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.012,
            f"{h:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
for bar in bars_f1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.012,
            f"{h:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_ylabel("Score (higher = better retrieval)", fontsize=11)
ax.set_title("Retrieval Quality Comparison: B1 vs B2 vs HyperRAG\n"
             f"({len(qa_items)} HotpotQA multi-hop questions)", fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(system_labels, fontsize=10)
ax.legend(fontsize=10)

# Give a little headroom above the tallest bar for labels
max_val = max(max(em_vals), max(f1_vals))
ax.set_ylim(0, max_val * 1.25 + 0.05)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()  # inline display only — no file saving

## Conclusion

This tutorial walked through the complete HyperRAG pipeline — from raw HotpotQA questions to a three-way retrieval evaluation with live results.

**What we built:**
- A Wikipedia corpus derived from 500 HotpotQA supporting-fact references (~450 pages)
- A NetworkX directed hyperlink graph connecting those pages via their `<a href>` links
- A FAISS flat inner-product index over 384-dim sentence-transformer embeddings
- Three retrieval systems: B1 (plain-text RAG), B2 (HTML-preserving RAG), and HyperRAG (graph-expanded RAG)

**The core insight:** HyperRAG's 1-hop graph expansion adds a reasoning step that neither baseline has. By following hyperlinks from retrieved pages, it discovers pages that are *structurally adjacent* to relevant content but *semantically distant* from the question — exactly the kind of multi-hop connection that HotpotQA was designed to test. The hyperlink graph costs nothing extra to build: the edges are already embedded in the Wikipedia HTML fetched for the corpus.

**To run the full 50-question evaluation:**
```bash
python run_all_systems.py
```
Results are saved to `data/results.csv` and summarised in `README.md`.

**For academic context** on RAG, GraphRAG, and multi-hop QA, see `writing/related_work_draft.md`.